In [0]:
%run ../common/environmental_config

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_drivers"

In [0]:
df_drivers = spark.table(f"{catalog_name}.{silver_schema}.drivers")
df_ref_nationality_region = spark.table(f"{catalog_name}.{gold_schema}.ref_nationality_region")

In [0]:
df_dim_drivers = (
    df_drivers.join(df_ref_nationality_region, df_drivers.nationality == df_ref_nationality_region.nationality, "left")
    .select(
        df_drivers.driver_id,
        df_drivers.driver_name,
        df_drivers.date_of_birth,
        df_drivers.nationality,
        df_ref_nationality_region.region.alias('nationality_region'))
)

In [0]:
#write to gold table
(df_dim_drivers
 .write
 .format("delta")
 .mode("overwrite")
 .saveAsTable(target_table))